# LSD1 structure-based screen

Ligand-based screening came back empty across 1.1M compounds from ChEMBL and
ZINC: the applicability domain rejected 91% of catalogue chemistry, and nothing
scored above p_active 0.25. That result stands - novel LSD1 chemotypes don't
resemble the known ones closely enough for a similarity-driven model to find
them.

Docking has no such limitation: it scores against the protein structure, so
chemistry unlike anything in ChEMBL is fair game.

**Files:** `actives.csv` (smiles, pchembl) and `library.csv` (smiles, catalog_id).
The first cell puts them on Google Drive, so a runtime restart no longer costs
you the uploads or the docking checkpoints.

**Order of operations matters.** Run the validation cell before the screen. If
known inhibitors don't outscore random compounds, nothing below is meaningful.

Runtime -> Change runtime type -> more CPUs. Vina is CPU-bound, GPU is unused.

In [ ]:
# ---- persistent working directory ------------------------------------------
# Colab wipes /content on every runtime restart. Google Drive survives, so we
# work directly on it: after os.chdir every read and write lands on Drive,
# including the docking checkpoints.
from google.colab import drive, files
import os

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/lsd1"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

need = [f for f in ("actives.csv", "library.csv") if not os.path.exists(f)]
if need:
    print("missing:", need, "-> upload now, this is the last time")
    up = files.upload()
    for name, data in up.items():
        clean = name.replace(" (1)", "").replace(" (2)", "")   # Colab suffixes dupes
        open(clean, "wb").write(data)
        print("  saved", clean)

print("\ncwd:", os.getcwd())
print("csv:", [f for f in os.listdir() if f.endswith(".csv")])

## 1. Environment

The `vina` PyPI package compiles from source against Boost and fails on recent
Python versions. The official release binary needs no compilation.

In [ ]:
# no compilation: the vina PyPI package builds from source and fails on recent
# Python. The official release binary just runs.
!pip install -q rdkit meeko gemmi
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64 -O /usr/local/bin/vina
!chmod +x /usr/local/bin/vina
!vina --version

import os, re, glob, time, shutil, subprocess, tempfile
import numpy as np, pandas as pd, gemmi
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem, Descriptors, Crippen, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from meeko import MoleculePreparation, PDBQTWriterLegacy

RDLogger.DisableLog("rdApp.*")
FPGEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

PDB_ID  = "2V1D"    # LSD1/CoREST. Alternatives: 2Z5U, 2UXX, 6TUZ
EXHAUST = 8         # 8 to triage, 16-32 for the finalists
BOX     = 22.0      # angstroms, cubic
N_DOCK  = 300       # start small. Raise once validation passes.
NCPU    = os.cpu_count()
print(f"\n{NCPU} CPUs available")

## 2. Receptor

2V1D is the LSD1-CoREST complex. FAD marks the catalytic site; tranylcypromine-class
inhibitors bind covalently there, reversible ones sit in the adjacent substrate
channel. The assert at the end is deliberate - a truncated receptor PDBQT still
"works" and silently returns garbage scores.

In [ ]:
# ---- receptor --------------------------------------------------------------
if not os.path.exists(f"{PDB_ID}.cif"):
    os.system(f"wget -q https://files.rcsb.org/download/{PDB_ID}.cif")

st = gemmi.read_structure(f"{PDB_ID}.cif")
st.setup_entities()

SKIP = {"HOH", "SO4", "GOL", "EDO", "CL", "NA", "MG", "ZN", "PO4", "ACT", "PEG"}
hets = {}
for model in st:
    for ch in model:
        for res in ch:
            if res.het_flag == "H" and res.name not in SKIP:
                hets.setdefault(res.name, []).append(
                    np.array([[a.pos.x, a.pos.y, a.pos.z] for a in res]))
for name, arrs in hets.items():
    print(f"  {name}: {len(arrs)} copies, {len(arrs[0])} atoms")

# FAD marks the catalytic site; inhibitors bind at or beside it
LIG = "FAD" if "FAD" in hets else max(hets, key=lambda k: len(hets[k][0]))
centre = hets[LIG][0].mean(axis=0)
print(f"\npocket anchor: {LIG}   centre = {centre.round(2)}")

st.remove_ligands_and_waters()
st.remove_hydrogens()
st.remove_alternative_conformations()
st.write_pdb("receptor.pdb")

# PDBQT needs partial charges + AutoDock atom types
ok = False
try:
    from meeko import PDBQTReceptor
    PDBQTReceptor.from_pdb_file("receptor.pdb").write_pdbqt_file("receptor.pdbqt")
    ok = True
except Exception as e:
    print("meeko receptor path unavailable:", type(e).__name__)
if not ok:
    os.system("apt-get -qq install -y openbabel > /dev/null 2>&1")
    os.system("obabel receptor.pdb -O receptor.pdbqt -xr --partialcharge gasteiger 2>/dev/null")

sz = os.path.getsize("receptor.pdbqt") if os.path.exists("receptor.pdbqt") else 0
assert sz > 10_000, "receptor prep failed - stop here, everything downstream would be noise"
print(f"receptor.pdbqt: {sz//1024} KB")

## 3. Ligand prep and the docking call

The `vina` PyPI package compiles from source against Boost and fails on recent
Python versions, which is what broke the previous notebook. The official release
binary needs no compilation, so this calls it through subprocess instead.

The smoke test at the bottom tells you the per-compound cost, which is how you
size `N_DOCK`.

In [ ]:
# ---- ligand prep + docking wrapper -----------------------------------------
def to_pdbqt(smiles, seed=0xf00d):
    """SMILES -> 3D -> PDBQT string. Vina searches torsions itself, so one
    ETKDG conformer is enough."""
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return None
    m = Chem.AddHs(m)
    if AllChem.EmbedMolecule(m, randomSeed=seed, useRandomCoords=True) != 0:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(m, maxIters=400)
    except Exception:
        pass
    try:
        s, good, _ = PDBQTWriterLegacy.write_string(MoleculePreparation().prepare(m)[0])
        return s if good else None
    except Exception:
        return None


def dock_one(pdbqt_str, centre=None, box=None, exhaust=None, cpu=None, timeout=300):
    """Best affinity in kcal/mol (lower = better), or None."""
    centre = centre if centre is not None else globals()["centre"]
    box, exhaust = box or BOX, exhaust or EXHAUST
    with tempfile.TemporaryDirectory() as d:
        lig = f"{d}/lig.pdbqt"
        open(lig, "w").write(pdbqt_str)
        cmd = ["vina", "--receptor", "receptor.pdbqt", "--ligand", lig,
               "--out", f"{d}/out.pdbqt",
               "--center_x", f"{centre[0]:.3f}", "--center_y", f"{centre[1]:.3f}",
               "--center_z", f"{centre[2]:.3f}",
               "--size_x", str(box), "--size_y", str(box), "--size_z", str(box),
               "--exhaustiveness", str(exhaust), "--cpu", str(cpu or NCPU),
               "--seed", "42"]
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            return None
        m = re.search(r"^\s*1\s+(-?\d+\.\d+)", r.stdout, re.M)
        return float(m.group(1)) if m else None


# smoke test on tranylcypromine, the classic LSD1 chemotype
pq = to_pdbqt("NC1CC1c1ccccc1")
print("ligand prep:", "OK" if pq else "FAILED")
t = time.time(); a = dock_one(pq); dt = time.time() - t
print(f"dock: {a} kcal/mol in {dt:.1f}s  ->  ~{dt*N_DOCK/60:.0f} min for {N_DOCK} compounds")

## 4. Validation - do not skip

Twenty known potent inhibitors versus twenty random library compounds. Actives
should score roughly 1-2 kcal/mol better. Takes a few minutes and saves you from
screening against a broken box.

In [ ]:
# ---- VALIDATE BEFORE SCREENING ---------------------------------------------
# Dock known potent inhibitors and random library compounds. If the knowns
# don't score better, the box or the receptor is wrong and every number below
# is noise. This is the step people skip and then publish nonsense.
act = pd.read_csv("actives.csv")
known = act.nlargest(20, "pchembl").smiles.tolist()
decoy = pd.read_csv("library.csv").sample(20, random_state=0).smiles.tolist()

def dock_set(smis, label):
    out = []
    for s in smis:
        pq = to_pdbqt(s)
        if pq is None:
            continue
        a = dock_one(pq)
        if a is not None:
            out.append(a)
    print(f"  {label:<18} n={len(out):2d}  mean {np.mean(out):6.2f}  best {np.min(out):6.2f}")
    return np.array(out)

t0 = time.time()
A = dock_set(known, "known actives")
D = dock_set(decoy, "random library")

from scipy.stats import mannwhitneyu
sep = D.mean() - A.mean()
p = mannwhitneyu(A, D, alternative="less").pvalue
print(f"\nseparation {sep:+.2f} kcal/mol   p={p:.3g}   ({(time.time()-t0)/60:.1f} min)")
if sep < 0.8:
    print("  FAIL: actives are not separating. Do not screen.")
    print("  Try: a different PDB (2Z5U, 6TUZ), a larger BOX, or move the centre")
    print("  toward the substrate channel rather than the FAD itself.")
else:
    print("  PASS: setup discriminates. Proceed to the screen.")

## 5. Screen

Checkpoints `docked.csv` every 25 compounds - Colab disconnects after about
90 minutes idle, and you don't want to lose a three-hour run.

In [ ]:
# ---- screen ----------------------------------------------------------------
lib = pd.read_csv("library.csv")
if "p_active" in lib.columns:                 # triage with the QSAR model if present
    lib = lib.sort_values("p_active", ascending=False)
lib = lib.head(N_DOCK).reset_index(drop=True)
print(f"docking {len(lib):,} compounds, exhaustiveness={EXHAUST}")

rows, t0 = [], time.time()
for i, r in lib.iterrows():
    pq = to_pdbqt(r.smiles)
    if pq is None:
        continue
    a = dock_one(pq)
    if a is None:
        continue
    rows.append({"smiles": r.smiles, "catalog_id": r.catalog_id, "affinity": a})
    if (i + 1) % 25 == 0:
        el = time.time() - t0
        print(f"  {i+1}/{len(lib)}  {el/60:.1f} min  "
              f"eta {el/(i+1)*(len(lib)-i-1)/60:.0f} min", flush=True)
        pd.DataFrame(rows).to_csv("docked.csv", index=False)   # checkpoint

dock = pd.DataFrame(rows).sort_values("affinity").reset_index(drop=True)
dock.to_csv("docked.csv", index=False)
print(f"\ndocked {len(dock):,}   best {dock.affinity.min():.2f}   "
      f"median {dock.affinity.median():.2f} kcal/mol")

## 6. Shortlist

Two filters do the work:

- **Ligand efficiency** >= 0.30. Without it your top hits are just the biggest
  molecules in the library.
- **One compound per scaffold**, so a single bad assumption doesn't take out all
  fifty picks.

`max_tanimoto` is reported but not filtered on - a hit far from known actives is
exactly what this notebook exists to find, and it's also the hit most likely to
be a docking artefact. Which is what the next cell is for.

In [ ]:
# ---- shortlist -------------------------------------------------------------
# Vina rewards size: a floppy 45-heavy-atom molecule scores well without binding
# anything. Ligand efficiency (-affinity / heavy atoms) is the correction.
dms = [Chem.MolFromSmiles(s) for s in dock.smiles]
ok = [i for i, m in enumerate(dms) if m]
dock, dms = dock.iloc[ok].reset_index(drop=True), [dms[i] for i in ok]

act_fp = [FPGEN.GetFingerprint(Chem.MolFromSmiles(s))
          for s, p in zip(act.smiles, act.pchembl)
          if p >= 8.5 and Chem.MolFromSmiles(s)]
dfp = [FPGEN.GetFingerprint(m) for m in dms]

dock["max_tanimoto"] = [round(max(DataStructs.BulkTanimotoSimilarity(f, act_fp)), 3) for f in dfp]
dock["scaffold"] = [MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False) for m in dms]
dock["heavy"] = [m.GetNumHeavyAtoms() for m in dms]
dock["mw"]    = [round(Descriptors.MolWt(m), 1) for m in dms]
dock["logp"]  = [round(Crippen.MolLogP(m), 2) for m in dms]
dock["LE"]    = (-dock.affinity / dock.heavy).round(3)
dock["consensus"] = (dock.affinity.rank() + (-dock.LE).rank()) / 2

hits = dock[(dock.LE >= 0.30) & (dock.affinity <= -8.0)]
short = hits.sort_values("consensus").drop_duplicates("scaffold").head(50)
short.to_csv("docking_shortlist.csv", index=False)

print(f"affinity <= -8.0 and LE >= 0.30: {len(hits):,} of {len(dock):,}")
print(f"shortlist (one per scaffold): {len(short)}")
print(f"novel chemistry: {(short.max_tanimoto < 0.4).sum()} of {len(short)} "
      f"below 0.4 Tanimoto to known actives\n")
print(short.head(15)[["catalog_id","affinity","LE","max_tanimoto","mw","logp"]].to_string(index=False))

## 7. Inspect the poses

The single most useful sanity check. If the top compound isn't contacting FAD or
the substrate channel residues, the score is an artefact no matter how good the
number looks. Re-docks at exhaustiveness 32 for a cleaner pose.

In [ ]:
# ---- look at the poses before believing anything ---------------------------
!pip install -q py3Dmol
import py3Dmol

def show(idx=0, exhaust=32):
    """Re-dock one shortlist compound at high exhaustiveness and view the pose."""
    row = short.iloc[idx]
    pq = to_pdbqt(row.smiles)
    with tempfile.TemporaryDirectory() as d:
        lig, out = f"{d}/l.pdbqt", f"{d}/o.pdbqt"
        open(lig, "w").write(pq)
        subprocess.run(["vina", "--receptor", "receptor.pdbqt", "--ligand", lig, "--out", out,
                        "--center_x", f"{centre[0]:.3f}", "--center_y", f"{centre[1]:.3f}",
                        "--center_z", f"{centre[2]:.3f}",
                        "--size_x", str(BOX), "--size_y", str(BOX), "--size_z", str(BOX),
                        "--exhaustiveness", str(exhaust), "--cpu", str(NCPU)],
                       capture_output=True, text=True)
        pose = open(out).read()

    v = py3Dmol.view(width=760, height=520)
    v.addModel(open("receptor.pdb").read(), "pdb")
    v.setStyle({"cartoon": {"color": "lightgrey"}})
    v.addModel(pose, "pdbqt")
    v.setStyle({"model": 1}, {"stick": {"colorscheme": "cyanCarbon"}})
    v.zoomTo({"model": 1})
    print(f"{row.catalog_id}  affinity {row.affinity}  LE {row.LE}  "
          f"Tanimoto {row.max_tanimoto}")
    return v.show()

show(0)

## Then what

Docking enrichment is typically 5-20x over random and pose prediction is more
reliable than affinity prediction, so treat this as a hypothesis list, not a
result. The honest sequence from here:

1. Eyeball the top 15 poses. Discard anything not making contact.
2. Re-dock the survivors at exhaustiveness 32 with three different seeds. Keep
   what's stable across runs.
3. Order 30-50 from the survivors.
4. Submit to the free NCI-60 screen, and separately test selectivity against
   BEAS-2B or MRC-5 - NCI-60 has no normal cell lines.

Expect single-digit percent hit rate. That's what a working campaign looks
like.